# 04 - 性能基准测试

本教程介绍如何测试RAG、Agent和多模态系统的性能。

## 学习目标
- 掌握延迟测试方法
- 理解准确率指标
- 学会吞吐量测试

## 1. 环境准备

In [ ]:
import sys
sys.path.append('../src')

import time
import numpy as np
from rag_benchmark import RAGBenchmark, RAGMetrics
from agent_benchmark import AgentBenchmark, AgentMetrics
from multimodal_benchmark import MultimodalBenchmark, MultimodalMetrics

## 2. RAG性能测试

In [ ]:
# 创建RAG基准测试器
rag_benchmark = RAGBenchmark(warmup_runs=2)

# 模拟检索函数
def mock_retrieve(query):
    time.sleep(0.005)  # 模拟5ms延迟
    return ["doc1", "doc2", "doc3"]

# 测试查询
queries = ["机器学习", "深度学习", "神经网络", "自然语言处理"]

In [ ]:
# 延迟测试
result = rag_benchmark.run_latency_test(mock_retrieve, queries, num_runs=5)

print("延迟测试结果:")
print(f"  平均延迟: {result.metrics.retrieval_latency_ms:.2f}ms")
print(f"  P50延迟: {result.details['p50_ms']:.2f}ms")
print(f"  P95延迟: {result.details['p95_ms']:.2f}ms")
print(f"  P99延迟: {result.details['p99_ms']:.2f}ms")

In [ ]:
# 吞吐量测试
result = rag_benchmark.run_throughput_test(mock_retrieve, queries, duration_seconds=2.0)

print("吞吐量测试结果:")
print(f"  QPS: {result.metrics.throughput_qps:.1f}")
print(f"  总查询数: {result.num_queries}")
print(f"  测试时长: {result.duration_seconds:.2f}s")

In [ ]:
# 准确率测试
def mock_retrieve_ids(query):
    return ["doc1", "doc2", "doc3", "doc4", "doc5"]

queries = ["q1", "q2", "q3"]
ground_truth = [
    ["doc1", "doc2"],
    ["doc2", "doc3", "doc4"],
    ["doc1", "doc5"],
]

result = rag_benchmark.run_accuracy_test(mock_retrieve_ids, queries, ground_truth, k=5)

print("准确率测试结果:")
print(f"  Recall@5: {result.metrics.recall_at_k:.2%}")
print(f"  Precision@5: {result.metrics.precision_at_k:.2%}")
print(f"  MRR: {result.metrics.mrr:.3f}")

## 3. Agent性能测试

In [ ]:
# 创建Agent基准测试器
agent_benchmark = AgentBenchmark()

# 模拟Agent执行
def mock_agent(task):
    time.sleep(0.01)
    return {
        "success": np.random.random() > 0.2,
        "steps": np.random.randint(2, 6),
        "tool_calls": np.random.randint(1, 4),
    }

tasks = ["任务1", "任务2", "任务3", "任务4", "任务5"]

In [ ]:
# 任务完成测试
result = agent_benchmark.run_task_completion_test(mock_agent, tasks)

print("Agent性能测试结果:")
print(f"  成功率: {result.metrics.success_rate:.1%}")
print(f"  平均步数: {result.metrics.avg_steps:.1f}")
print(f"  平均工具调用: {result.metrics.avg_tool_calls:.1f}")
print(f"  平均延迟: {result.metrics.avg_latency_ms:.2f}ms")

In [ ]:
# 查看每个任务的结果
print("\n各任务详情:")
for tr in result.task_results:
    status = "成功" if tr.success else "失败"
    print(f"  {tr.task_id}: {status}, {tr.steps}步, {tr.latency_ms:.1f}ms")

## 4. 多模态性能测试

In [ ]:
# 创建多模态基准测试器
mm_benchmark = MultimodalBenchmark(warmup_runs=2)

# 模拟编码函数
def img_encode(img):
    time.sleep(0.002)
    return np.random.rand(256)

def txt_encode(txt):
    time.sleep(0.001)
    return np.random.rand(256)

# 测试数据
images = [np.random.rand(64, 64, 3) for _ in range(10)]
texts = [f"文本{i}" for i in range(10)]

In [ ]:
# 编码性能测试
result = mm_benchmark.run_encoding_test(img_encode, txt_encode, images, texts)

print("编码性能测试结果:")
print(f"  图像编码: {result.metrics.image_encode_ms:.2f}ms")
print(f"  文本编码: {result.metrics.text_encode_ms:.2f}ms")
print(f"  图像P95: {result.details['image_p95_ms']:.2f}ms")

In [ ]:
# 相似度测试
def similarity(img, txt):
    return np.random.uniform(0.6, 0.9)

pairs = [(np.random.rand(32, 32, 3), f"描述{i}") for i in range(5)]
result = mm_benchmark.run_similarity_test(similarity, pairs)

print("相似度测试结果:")
print(f"  平均相似度: {result.metrics.image_text_similarity:.3f}")
print(f"  相似度标准差: {result.details['sim_std']:.3f}")

## 5. 测试报告

In [ ]:
# 查看所有测试结果
print("RAG测试摘要:")
print(rag_benchmark.summary())

print("\nAgent测试摘要:")
print(agent_benchmark.summary())

print("\n多模态测试摘要:")
print(mm_benchmark.summary())

## 6. 练习

1. 修改模拟函数的延迟，观察性能变化
2. 增加测试数据量，比较结果
3. 尝试不同的k值，观察准确率变化

In [ ]:
# 练习空间


## 总结

本教程介绍了:
- RAGBenchmark: 延迟/吞吐量/准确率测试
- AgentBenchmark: 任务完成率/步数效率测试
- MultimodalBenchmark: 编码/检索/相似度测试

下一步: 学习端到端系统集成